# Basic

In [13]:
%load_ext autoreload
%autoreload all

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [14]:
import os
import pickle
import multiprocessing as mp
import tqdm
import numpy as np

import polars as pl

import src.graph_tokenizer_gd_tree_dev.config as config
from src.graph_tokenizer_gd_tree_dev.phenotype_clustering import (
    cluster_and_characterize,
    init_cluster_worker,
    cluster_worker,
)

# load id_to_label + the same method specs 6.3 used (to recover each tok_i's real concept id)

In [15]:
with open(config.ProcessedGraph().id_to_label, "rb") as f:
    id_to_label = pickle.load(f)

method_specs = {
    "lam_0.6": config.IACandidateLists().path_greedy_tree + "0.6.parquet",
    "lam_0.8": config.IACandidateLists().path_greedy_tree + "0.8.parquet",
    "lam_1.0": config.IACandidateLists().path_greedy_tree + "1.0.parquet",
    "personalized_pagerank": config.IACandidateLists().personalized_pagerank,
    "discrete_set_cover": config.IACandidateLists().discrete_set_cover,
}
Ks = np.arange(100, 6000, 100)          # same grid as 6.1's selection sweep
combos = [(name, k) for name in method_specs for k in Ks]

# clustering + parallel worker now live in `phenotype_clustering.py`, not this notebook

Moved out of the notebook so `mp.get_context("forkserver")` can import them by module path
instead of forking the live, multi-threaded Jupyter kernel directly (see the module
docstring for why that was hanging).

# no_tokenizer clustered first (it's the reference), then every (method, k) combo, reporting
# ARI/NMI of each combo's cluster assignment against the no_tokenizer assignment -- do the reduced
# vocabularies rediscover the same patient structure as clustering on everything, or a different one?

In [ ]:
os.makedirs(config.IAFeatures().path + "phenotype_clusters/", exist_ok=True)
N_SEEDS = 5

# --- reference: no_tokenizer, clustered on the raw full concept vocabulary, averaged over 5 seeds.
# assign_ref_dict is fixed at the seed=0 assignment -- every combo's 5 seeds (below) compare back
# to this one fixed reference, so ARI/NMI variance reflects the combo's own stability, not the
# reference's. ---
df_feat_full = pl.read_parquet(f"{config.IAFeatures().path}no_tokenizer_kfull.parquet")
T_full = pl.read_parquet(f"{config.IAFeatures().path}no_tokenizer_vocab.parquet").sort("index")["token"].to_list()

ref_n_clusters, ref_silhouettes, df_assign_ref = [], [], None
for seed in range(N_SEEDS):
    n_clusters_ref, silhouette_ref, df_clusters_ref, df_assign_ref_seed = cluster_and_characterize(df_feat_full, T_full, id_to_label, seed=seed)
    ref_n_clusters.append(n_clusters_ref)
    ref_silhouettes.append(silhouette_ref)
    if seed == 0:
        df_assign_ref = df_assign_ref_seed
        df_clusters_ref.write_parquet(f"{config.IAFeatures().path}phenotype_clusters/no_tokenizer_kfull.parquet")

assign_ref_dict = dict(zip(df_assign_ref["unique_patient_id"].to_list(), df_assign_ref["cluster"].to_list()))

quality_rows = [{
    "method": "no_tokenizer",
    "k": len(T_full),
    "n_clusters_mode": max(set(ref_n_clusters), key=ref_n_clusters.count),
    "silhouette_mean": float(np.mean(ref_silhouettes)),
    "silhouette_std": float(np.std(ref_silhouettes)),
    "ari_vs_no_tokenizer_mean": 1.0,   # trivially perfect agreement with itself
    "ari_vs_no_tokenizer_std": 0.0,
    "nmi_vs_no_tokenizer_mean": 1.0,
    "nmi_vs_no_tokenizer_std": 0.0,
}]

# --- every (method, k) combo, in parallel: 5 seeds x (PCA + KMeans(x6) + silhouette(x6)) per combo,
# averaged inside cluster_worker (see phenotype_clustering.py). forkserver, not the default fork:
# forks worker processes from a clean, single-threaded server process instead of this live,
# multi-threaded Jupyter kernel -- avoids the deadlock fork caused. ---
n_workers = min(os.cpu_count(), len(combos))
ctx = mp.get_context("forkserver")
with ctx.Pool(n_workers, initializer=init_cluster_worker, initargs=(id_to_label, method_specs, assign_ref_dict)) as pool:
    for row in tqdm.tqdm(pool.imap_unordered(cluster_worker, combos), total=len(combos)):
        quality_rows.append(row)

quality_df = pl.DataFrame(quality_rows).sort("silhouette_mean", descending=True)
quality_df.write_parquet(f"{config.IAFeatures().path}phenotype_clusters/_quality_summary.parquet")
quality_df

 20%|█▉        | 58/295 [54:49<3:17:33, 50.02s/it]  

# does better conciseness (soft, higher lambda) also mean more interpretable clusters?

Sorted by silhouette_mean -- if the same lambda range that won on `distance_score`/`conciseness`
intrinsically (see 6.2) also wins here, that's a second, independent validation of the same
design choice, not just a repeat of the classification result. Error bars are std across the
5 seeds (see `cluster_worker` in `phenotype_clustering.py`) -- if error bars between two
methods overlap heavily at a given `k`, don't read the ranking there as a real difference.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(21, 5))

for metric, ax, title in zip(
    ["silhouette", "ari_vs_no_tokenizer", "nmi_vs_no_tokenizer"],
    axes,
    ["silhouette (internal quality)", "ARI vs. no_tokenizer", "NMI vs. no_tokenizer"],
):
    mean_col, std_col = f"{metric}_mean", f"{metric}_std"
    for method_name in method_specs:
        sub = quality_df.filter(pl.col("method") == method_name).sort("k")
        ax.errorbar(sub["k"], sub[mean_col], yerr=sub[std_col], marker="o", capsize=3, label=method_name)
    if metric != "silhouette":
        ax.set_ylim(0, 1.05)  # ARI/NMI are bounded; no_tokenizer's own point is trivially 1.0, not plotted here
    else:
        no_tok_val = quality_df.filter(pl.col("method") == "no_tokenizer")[mean_col][0]
        ax.axhline(no_tok_val, color="gray", linestyle="--", label="no_tokenizer (full vocab)")
    ax.set_xlabel("k")
    ax.set_title(title)
    ax.grid(alpha=0.3)

axes[0].set_ylabel("score (mean +/- std over 5 seeds)")
axes[0].legend(fontsize=8)
fig.suptitle("IA case study -- phenotype cluster quality vs k", fontsize=14)
fig.tight_layout()
plt.show()

# inspect the top clusters for your primary method (lam=0.8) at one k

In [ ]:
pl.read_parquet(f"{config.IAFeatures().path}phenotype_clusters/lam_0.8_k1000.parquet")